# Fase 8: Guardar modelo y preparar base para producción

En esta fase se entrena el modelo final seleccionado, se guarda junto con su pipeline de transformaciones y se deja una base funcional para inferencia sobre nuevos datos.

## Objetivo de la fase

El objetivo de esta fase es conectar el experimento de Machine Learning con una posible integración real.

Para ello se realizará lo siguiente:

- entrenar el modelo final dentro de un pipeline reproducible;
- guardar el modelo con joblib;
- documentar cómo cargarlo;
- preparar un ejemplo de predicción sobre nuevos datos.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-features.csv"

df = pd.read_csv(path)

print("Dataset cargado correctamente")
print(df.shape)
df.head()

Dataset cargado correctamente
(110526, 22)


,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,...,waiting_days,scheduled_weekday,appointment_weekday,is_same_day,age_group,has_comorbidity,risk_score,is_child_or_senior,appointment_month,schedule_hour
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,...,0,4,4,1,senior,1,1,1,4,18
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,...,0,4,4,1,adult,0,0,0,4,16
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,...,0,4,4,1,senior,0,0,1,4,16
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,...,0,4,4,1,child,0,0,1,4,17
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,...,0,4,4,1,adult,1,2,0,4,16


In [3]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], errors='coerce')
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], errors='coerce')

## Variables de entrada y salida

Se define la variable objetivo y se eliminan columnas que no se utilizarán directamente en el modelo final.

In [4]:
y = df['No_show']

X = df.drop(columns=[
    'No_show',
    'ScheduledDay',
    'AppointmentDay'
])

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (110526, 19)
Shape de y: (110526,)


In [5]:
categorical_cols = ['Neighbourhood', 'age_group']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Variables categóricas:", categorical_cols)
print("Variables numéricas:", numeric_cols)

Variables categóricas: ['Neighbourhood', 'age_group']
Variables numéricas: ['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'waiting_days', 'scheduled_weekday', 'appointment_weekday', 'is_same_day', 'has_comorbidity', 'risk_score', 'is_child_or_senior', 'appointment_month', 'schedule_hour']


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

## Separación de entrenamiento y prueba

Se conserva partición estratificada para respetar la proporción de clases.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (88420, 19)
Test size: (22106, 19)


## Modelo final seleccionado

Se utiliza Logistic Regression con `class_weight='balanced'`, ya que fue el modelo con mejor recall y mejor F1-score para la clase minoritaria.

In [8]:
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

final_model.fit(X_train, y_train)

print("Modelo final entrenado correctamente")

Modelo final entrenado correctamente


In [9]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.93      0.51      0.66     17642
           1       0.30      0.84      0.45      4464

    accuracy                           0.58     22106
   macro avg       0.62      0.68      0.55     22106
weighted avg       0.80      0.58      0.62     22106

Confusion Matrix:
 [[8997 8645]
 [ 696 3768]]
Accuracy: 0.5774450375463676
Precision: 0.30355272697977925
Recall: 0.8440860215053764
F1: 0.44652485631332584
ROC-AUC: 0.7223111003738636


## Carpeta de salida

Se crea la carpeta `models` si no existe, para guardar el pipeline final.

In [10]:
models_dir = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\models"
os.makedirs(models_dir, exist_ok=True)

print("Carpeta models lista:")
print(models_dir)

Carpeta models lista:
C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\models


In [11]:
model_path = os.path.join(models_dir, "no_show_model.joblib")
joblib.dump(final_model, model_path)

print("Modelo guardado en:")
print(model_path)

Modelo guardado en:
C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\models\no_show_model.joblib


In [12]:
feature_columns_path = os.path.join(models_dir, "feature_columns.joblib")
joblib.dump(X.columns.tolist(), feature_columns_path)

print("Columnas de entrada guardadas en:")
print(feature_columns_path)
print(X.columns.tolist())

Columnas de entrada guardadas en:
C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\models\feature_columns.joblib
['Gender', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'waiting_days', 'scheduled_weekday', 'appointment_weekday', 'is_same_day', 'age_group', 'has_comorbidity', 'risk_score', 'is_child_or_senior', 'appointment_month', 'schedule_hour']


## Ejemplo de inferencia sobre un nuevo registro

Se construye un ejemplo manual de entrada con la misma estructura esperada por el modelo.

In [13]:
new_data = pd.DataFrame([{
    'Gender': 0,
    'Age': 45,
    'Neighbourhood': 'JARDIM DA PENHA',
    'Scholarship': 0,
    'Hipertension': 1,
    'Diabetes': 0,
    'Alcoholism': 0,
    'Handcap': 0,
    'SMS_received': 1,
    'waiting_days': 12,
    'scheduled_weekday': 1,
    'appointment_weekday': 3,
    'is_same_day': 0,
    'age_group': 'adult',
    'has_comorbidity': 1,
    'risk_score': 1,
    'is_child_or_senior': 0,
    'appointment_month': 5,
    'schedule_hour': 10
}])

new_data

,Gender,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,waiting_days,scheduled_weekday,appointment_weekday,is_same_day,age_group,has_comorbidity,risk_score,is_child_or_senior,appointment_month,schedule_hour
0,0,45,JARDIM DA PENHA,0,1,0,0,0,1,12,1,3,0,adult,1,1,0,5,10


In [14]:
pred_class = final_model.predict(new_data)[0]
pred_prob = final_model.predict_proba(new_data)[0, 1]

print("Predicción (0=asiste, 1=no asiste):", pred_class)
print("Probabilidad de no asistencia:", round(pred_prob, 4))

Predicción (0=asiste, 1=no asiste): 0
Probabilidad de no asistencia: 0.4758


## Carga del modelo guardado

Se valida que el modelo pueda cargarse desde disco y utilizarse nuevamente.

In [15]:
loaded_model = joblib.load(model_path)

loaded_pred_class = loaded_model.predict(new_data)[0]
loaded_pred_prob = loaded_model.predict_proba(new_data)[0, 1]

print("Predicción con modelo cargado:", loaded_pred_class)
print("Probabilidad con modelo cargado:", round(loaded_pred_prob, 4))

Predicción con modelo cargado: 0
Probabilidad con modelo cargado: 0.4758


## Documentación del modelo guardado

Archivos generados:

- `models/no_show_model.joblib`: pipeline completo con transformaciones y modelo entrenado.
- `models/feature_columns.joblib`: lista de columnas esperadas por el modelo.

Ventaja de guardar el pipeline completo:
- no es necesario rehacer manualmente el escalado o codificación;
- el modelo puede recibir nuevos datos directamente con el mismo formato de entrada;
- se reduce el riesgo de inconsistencias entre entrenamiento e inferencia.

## Hallazgos finales de la fase

En esta fase se logró entrenar y guardar el modelo final junto con su pipeline de transformación, permitiendo reutilizarlo de forma consistente fuera del notebook.

Además, se validó el proceso de inferencia tanto desde el notebook como desde un script independiente, confirmando que el flujo funciona correctamente y puede servir como base para una integración futura en el sistema Healthy Reminder.

## Conclusión de la Fase 8

El modelo final fue preparado con éxito para su reutilización en un entorno más cercano a producción.

Se guardó el pipeline completo con `joblib`, se documentó la estructura de entrada esperada y se construyó un script básico de predicción para nuevos registros.

Esto representa un paso importante para conectar el experimento de Machine Learning con una posible integración real dentro del proyecto Healthy Reminder.